# 氷結晶粒径予測パイプライン
ResNet18 を用いて顕微鏡クロップ画像から円相当径（µm）を回帰予測する。

| セル | 内容 |
|------|------|
| Step 0 | セットアップ（ライブラリ・定数・Driveマウント） |
| Step 1 | データセット作成（クロップ画像 + CSV） |
| Step 2 | 学習準備（Dataset・DataLoader・モデル定義） |
| Step 3 | 学習ループ（20 epoch、best モデル自動保存） |
| Step 4 | 評価（R²・MAE・RMSE）＋ ダウンロード |

> **注意**: ランタイム → GPU を選択してから実行してください。

## Step 0: セットアップ

In [ ]:
import os, re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet18
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

from google.colab import drive
drive.mount('/content/drive')

# ===== パス設定（ここを実際のパスに変更） =====
IMAGE_ROOT  = "/content/drive/MyDrive/研究/画像データ"
CSV_ROOT    = "/content/drive/MyDrive/研究/Excelデータ"
OUTPUT_DIR  = "/content/crops"
DATASET_CSV = "/content/dataset.csv"
MODEL_PATH  = "/content/best_model.pth"

# ===== スケール設定 =====
MAG_TO_UM_PER_PIXEL = {40: 0.088725, 20: 0.17353, 10: 0.34392}
BASE_UM_PER_PIXEL   = MAG_TO_UM_PER_PIXEL[40]

COL_X1          = "矩形領域(左上:x)"
COL_Y1          = "矩形領域(左上:y)"
COL_X2          = "矩形領域(右下:x)"
COL_Y2          = "矩形領域(右下:y)"
COL_DIAMETER    = "円相当径"
IMG_EXTS        = (".bmp", ".tif", ".tiff", ".jpg", ".png")
EXCLUDE_MINUTES = {"0"}

PAD_SIZE = 512
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"デバイス: {device}")

## Step 1: データセット作成
`OUTPUT_DIR` にクロップ画像を、`DATASET_CSV` にラベル CSV を出力する。
**既に実行済みの場合はスキップ可。**

In [ ]:
def normalize_folder_name(name):
    return re.sub(r"[（）()・\s]", "", name)

def get_minute_mag_key(filename):
    m = re.search(r"(\d+)分.*?(\d+)倍", filename)
    return m.groups() if m else None

def find_diameter_col(columns):
    for c in columns:
        if COL_DIAMETER in str(c):
            return c
    raise ValueError(f"円相当径列が見つかりません: {list(columns)}")

def make_dataset():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    img_dirs = {normalize_folder_name(d): d
                for d in os.listdir(IMAGE_ROOT)
                if os.path.isdir(os.path.join(IMAGE_ROOT, d))}
    csv_dirs = {normalize_folder_name(d): d
                for d in os.listdir(CSV_ROOT)
                if os.path.isdir(os.path.join(CSV_ROOT, d))}

    common = set(img_dirs) & set(csv_dirs)
    print(f"画像フォルダ: {len(img_dirs)}件  CSVフォルダ: {len(csv_dirs)}件  対応: {len(common)}件")

    rows, crop_id = [], 0

    for key in sorted(common):
        image_dir = os.path.join(IMAGE_ROOT, img_dirs[key])
        csv_dir   = os.path.join(CSV_ROOT,   csv_dirs[key])
        print(f"\n=== {img_dirs[key]} ===")

        images = defaultdict(list)
        for f in os.listdir(image_dir):
            if f.lower().endswith(IMG_EXTS):
                k = get_minute_mag_key(f)
                if k:
                    images[k].append(f)

        csvs = defaultdict(list)
        for f in os.listdir(csv_dir):
            if f.lower().endswith(".csv"):
                k = get_minute_mag_key(f)
                if k:
                    csvs[k].append(f)

        for k in sorted(set(images) & set(csvs)):
            minute, _ = k
            if minute in EXCLUDE_MINUTES:
                print(f"  [除外] {minute}分")
                continue

            for img_f, csv_f in zip(sorted(images[k]), sorted(csvs[k])):
                image_path = os.path.join(image_dir, img_f)
                csv_path   = os.path.join(csv_dir,   csv_f)

                m = re.search(r"(\d+)倍", img_f)
                mag = int(m.group(1)) if m else None
                if mag not in MAG_TO_UM_PER_PIXEL:
                    continue
                um_per_pixel = MAG_TO_UM_PER_PIXEL[mag]

                try:
                    df  = pd.read_csv(csv_path, encoding="cp932")
                    dcol = find_diameter_col(df.columns)
                    img  = Image.open(image_path)
                except Exception as e:
                    print(f"  [スキップ] {e}")
                    continue

                n_crops = 0
                for _, crystal in df.iterrows():
                    try:
                        x1 = int(crystal[COL_X1]); y1 = int(crystal[COL_Y1])
                        x2 = int(crystal[COL_X2]); y2 = int(crystal[COL_Y2])
                        d  = float(crystal[dcol])
                    except (KeyError, ValueError, TypeError):
                        continue

                    left, right = sorted([x1, x2])
                    top, bottom = sorted([y1, y2])
                    if right <= left or bottom <= top:
                        continue

                    crop  = img.crop((left, top, right, bottom))
                    scale = um_per_pixel / BASE_UM_PER_PIXEL
                    crop  = crop.resize((max(1, round(crop.width  * scale)),
                                         max(1, round(crop.height * scale))))

                    crop_path = os.path.join(OUTPUT_DIR, f"crop_{crop_id:06d}.png")
                    crop.save(crop_path)

                    rows.append({"filepath": crop_path,
                                 "label_um": d * um_per_pixel,
                                 "source_folder": img_dirs[key],
                                 "source_image":  img_f})
                    crop_id += 1
                    n_crops += 1

                print(f"  {img_f} → {n_crops}件")

    out_df = pd.DataFrame(rows)
    out_df.to_csv(DATASET_CSV, index=False, encoding="utf-8-sig")
    print(f"\n完了: {len(out_df)}件 → {DATASET_CSV}")

make_dataset()

## Step 2: 学習準備（Dataset・DataLoader・モデル）

In [ ]:
# ===== 前処理 =====
def pad_to_square(img):
    w, h = img.size
    if w > PAD_SIZE:
        left = (w - PAD_SIZE) // 2
        img  = img.crop((left, 0, left + PAD_SIZE, h))
        w    = PAD_SIZE
    if h > PAD_SIZE:
        top = (h - PAD_SIZE) // 2
        img  = img.crop((0, top, w, top + PAD_SIZE))
        h    = PAD_SIZE
    padded = Image.new(img.mode, (PAD_SIZE, PAD_SIZE), 0)
    padded.paste(img, ((PAD_SIZE - w) // 2, (PAD_SIZE - h) // 2))
    return padded

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Lambda(pad_to_square),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Lambda(pad_to_square),
    transforms.ToTensor(),
])

# ===== Dataset =====
class CrystalDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(row["label_um"], dtype=torch.float32)

# ===== CSV 読み込み・検証（1回だけ） =====
raw_df = pd.read_csv(DATASET_CSV)
valid_idx = []
for i, fp in enumerate(raw_df["filepath"]):
    try:
        with Image.open(fp) as im:
            im.verify()
        valid_idx.append(i)
    except Exception:
        pass
removed = len(raw_df) - len(valid_idx)
if removed:
    print(f"壊れた画像を除外: {removed}件")
df_all = raw_df.iloc[valid_idx].reset_index(drop=True)
print(f"有効データ: {len(df_all)}件")

# ===== 8:2 分割 =====
torch.manual_seed(42)
perm    = torch.randperm(len(df_all)).tolist()
n_train = int(len(df_all) * 0.8)
train_df = df_all.iloc[perm[:n_train]].reset_index(drop=True)
test_df  = df_all.iloc[perm[n_train:]].reset_index(drop=True)
print(f"train: {len(train_df)}件 / test: {len(test_df)}件")

train_loader = DataLoader(CrystalDataset(train_df, train_transform),
                          batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(CrystalDataset(test_df,  test_transform),
                          batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# ===== モデル・損失・最適化 =====
model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)

criterion = nn.HuberLoss(delta=1.0)
mse_fn    = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
print("モデル準備完了")

## Step 3: 学習

In [ ]:
EPOCHS   = 20
best_mse = float("inf")

for ep in range(1, EPOCHS + 1):
    # --- train ---
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device).unsqueeze(1)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    train_mse = total_loss / len(train_df)

    # --- eval ---
    model.eval()
    t_mse = t_mae = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device).unsqueeze(1)
            out   = model(x)
            t_mse += mse_fn(out, y).item() * x.size(0)
            t_mae += (out - y).abs().sum().item()
    test_mse = t_mse / len(test_df)
    test_mae = t_mae / len(test_df)

    scheduler.step(test_mse)
    tag = ""
    if test_mse < best_mse:
        best_mse = test_mse
        torch.save(model.state_dict(), MODEL_PATH)
        tag = "  ← best 保存"
    print(f"Epoch {ep:2d}: train={train_mse:.4f}  test={test_mse:.4f}  MAE={test_mae:.4f} µm{tag}")

print(f"\n学習完了。best test MSE = {best_mse:.4f}")

## Step 4: 評価（R²）＋ ダウンロード

In [ ]:
from google.colab import files

# best モデルを読み込んで評価
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

preds, trues = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device)).squeeze(1).cpu().numpy()
        preds.extend(out.tolist())
        trues.extend(y.numpy().tolist())

preds = np.array(preds)
trues = np.array(trues)

ss_res = np.sum((trues - preds) ** 2)
ss_tot = np.sum((trues - trues.mean()) ** 2)
r2   = 1 - ss_res / ss_tot
mae  = np.mean(np.abs(trues - preds))
rmse = np.sqrt(np.mean((trues - preds) ** 2))

print(f"R²   = {r2:.4f}")
print(f"MAE  = {mae:.4f} µm")
print(f"RMSE = {rmse:.4f} µm")

# ダウンロード
files.download(MODEL_PATH)
files.download(DATASET_CSV)